<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 4, Direct Preference Optimization**

**Contexto**

Un modelo dentro de un pipeline le "habla" a un json.loads(), los modelos pequeños envuelven la respuesta en un json pero agregan una frase antes o cierran mal una llave, etc. Esto pude darnos problemas...

Lo que se hará en este notebook es que dada una noticia real, devolver exactamente esto y nada más, por ejemplo:
```json
{"topic": "Sports", "headline": "Federer wins fourth straight title"}
```

**cómo construimos los pares?**

El modelo intenta cada noticia tres veces con temperatura, nos quedamos con las noticias donde al menos un intento fue JSON válido y al menos uno no, y el par es:

* elegida = un intento propio que parsea
* descartada = un intento propio que no parsea

los dos salen de la misma política, con el mismo prompt, la única diferencia es
si un programa puede leerlos.

**métricas**

| métrica | qué es | qué esperamos |
|---|---|---|
| **validez** | fracción de salidas que json.loads acepta con las dos claves | que suba mucho |
| **exactitud del tema** | fracción que clasifica bien la noticia | que no baje |

La segunda es un guardarraíl: si el modelo aprende el formato a costa de clasificar peor no estamos obteniendo nada.

Instalamos versiones fijas que necesitamos

In [ ]:
!pip install -q -U "transformers>=4.45" "trl>=0.12" "peft>=0.13" datasets accelerate bitsandbytes

import torch, transformers, trl, peft

print("transformers", transformers.__version__, "· trl", trl.__version__, "· peft", peft.__version__)
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")


transformers 5.17.0 · trl 1.14.0 · peft 0.21.0
GPU Tesla T4


### **El modelo, el pedido y las dos mediciones**

El criterio de validez es el de un pipeline real: `json.loads()` sobre la salida cruda.
* una respuesta envuelta en `json` es inválida, aunque el contenido esté perfecto, porque el programa que la recibe cae
* se mide también una validez tolerante, que quita las cercas antes de parsear


In [ ]:
import gc
import re
import json as js
import random
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset

In [ ]:
MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
N_ENTRENA = 300 # noticias para armar los pares
K_INTENTOS = 3 # intentos por noticia, con temperatura
N_EVAL = 250 # noticias para medir, antes y despues

In [ ]:
TEMAS = ["World", "Sports", "Business", "Sci/Tech"]

In [ ]:
DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
USA_BF16, USA_FP16 = DTYPE == "bfloat16", DTYPE == "float16"

Acá tenemos QLoRA estándar:
* los pesos viven en 4 bits con cuantización nf4 (representa los pesos de un modelo usando solo 4 bits)
* La doble cuantización cuantiza también las escalas usadas para cuantizar los pesos, ahorrando aún más memoria
* el cómputo se desenrolla en fp16

los pesos base quedan congelados, lo único que se entrena son los adaptadores LoRA en fp16

In [ ]:
BNB = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=getattr(torch, DTYPE))

importante, con padding a la izquierda todas las filas terminan en el último token real del prompt, justo donde debe arrancar la generación

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tok.padding_side = "left"

In [ ]:
def cargar_base():
    return AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=BNB, device_map="auto")

In [ ]:
SISTEMA = (
    "You classify news snippets. Reply with a single JSON object and nothing else.\n"
    "Schema: {\"topic\": one of World | Sports | Business | Sci/Tech, "
    "\"headline\": a headline of at most 8 words}\n"
    "No markdown, no code fences, no explanation. Output must start with { and end with }."
)

* **tokenize=False** devuelve texto no ids, porque ese mismo texto se guarda como campo prompt del dataset de DPO y TRL lo tokeniza por su cuenta
* **add_generation_prompt=True** agrega la cabecera del turno del asistente

In [ ]:
def prompt_de(noticia):
    return tok.apply_chat_template(
        [{"role": "system", "content": SISTEMA},
         {"role": "user", "content": noticia}],
        tokenize=False, add_generation_prompt=True)

En el siguiente bloque se generan los lotes, importante destacar que cambiar tam cambia las semillas y por lo tanto las salidas muestreadas

In [ ]:
def generar_lote(modelo, textos, max_tokens=90, tam=32, temperatura=None, semilla=0):
    """Genera en lotes, siempre en modo evaluacion.

    Un modelo recien entrenado queda en modo entrenamiento, y ahi la dropout de
    LoRA sigue activa: la salida se degrada sin que nada lance un error.

    temperatura=None: codicioso y determinista para medir
    temperatura=0.8:  muestreo para construir los pares"""

    opciones = dict(max_new_tokens=max_tokens, pad_token_id=tok.pad_token_id)

    if temperatura is None:
        opciones["do_sample"] = False
    else:
        opciones.update(do_sample=True, temperature=temperatura, top_p=0.95)

    estaba = modelo.training
    modelo.eval()
    salidas = []

    # el try/finally garantiza que se restaure el estado original incluso si la generación explota
    try:
        for i in range(0, len(textos), tam):
            trozo = [prompt_de(t) for t in textos[i:i + tam]]
            e = tok(trozo, return_tensors="pt", padding=True).to(modelo.device)
            torch.manual_seed(semilla + i)
            with torch.no_grad():
                out = modelo.generate(**e, **opciones)

            # generate devuelve el prompt concatenado con la respuesta, se descartan
            # los primeros corte tokens para quedarse solo con lo generado
            corte = e.input_ids.shape[1]
            for fila in out:
                salidas.append(tok.decode(fila[corte:], skip_special_tokens=True).strip())
            print(f"  {min(i + tam, len(textos))}/{len(textos)}", end="\r")

    finally:
        modelo.train(estaba)

    print()
    return salidas

**Las dos mediciones:**

Una respuesta envuelta en json es inválida aunque el contenido sea perfecto, porque el programa que la recibe se cae. Además este bloque exige las dos claves y que el tema esté en el vocabulario permitido

In [ ]:
def valido(salida):
    # Lo que aceptaria una tuberia real: json.loads sobre la salida cruda
    try:
        d = js.loads(salida.strip())
    except Exception:
        return 0
    return int(isinstance(d, dict) and "topic" in d and "headline" in d
               and d["topic"] in TEMAS)

La función valido_tolerante quita las cercas antes de parsear, la distancia entre las dos métricas nos dice qué parte del problema es solo markdown y qué parte es JSON roto

In [ ]:
def sin_cerca(salida):
    # Quita json ... para medir cuanto del problema es solo la cerca
    m = re.search(r"```(?:json)?\s*(.+?)```", salida, re.S)
    return (m.group(1) if m else salida).strip()

def valido_tolerante(salida):
    return valido(sin_cerca(salida))

Esta función es para el guardarraíl, si el JSON no parsea, se busca cualquiera de las cuatro etiquetas dentro del texto crudo y se toma la que aparece primero (min sobre las tuplas ordena por posición), asi la exactitud se mide sobre las 250 salidas en ambos casos y las dos métricas quedan separadas

In [ ]:
def tema_de(salida):
    try:
        d = js.loads(sin_cerca(salida))
        if isinstance(d, dict) and d.get("topic") in TEMAS:
            return d["topic"]
    except Exception:
        pass

    bajo = salida.lower()
    posiciones = [(bajo.find(t.lower()), t) for t in TEMAS if t.lower() in bajo]
    return min(posiciones)[1] if posiciones else None

### **El dataset**

AG News:
* 120.000 noticias reales de agencia, etiquetadas en cuatro temas
* se usan fragmentos cortos
* la etiqueta sirve para el guardarrail de exactitud


In [ ]:
AG = load_dataset("fancyzhx/ag_news")

Este filtro de longitud saca los fragmentos truncados y los demasiado largos, que serían ruido

In [ ]:
def preparar(split, n, semilla):
    d = split.filter(lambda f: 80 < len(f["text"]) < 400).shuffle(seed=semilla)
    d = d.select(range(n))
    return [f["text"] for f in d], [TEMAS[f["label"]] for f in d]

In [ ]:
TEXTO_TRAIN, TEMA_TRAIN = preparar(AG["train"], N_ENTRENA, 11)
TEXTO_EVAL, TEMA_EVAL = preparar(AG["test"], N_EVAL, 7)

In [ ]:
print("noticias para armar pares:", len(TEXTO_TRAIN))
print("noticias para medir:", len(TEXTO_EVAL))
print("Ejemplo de una noticia:")
print(TEXTO_EVAL[0])
print("tema correcto:", TEMA_EVAL[0])

noticias para armar pares: 300
noticias para medir: 250
Ejemplo de una noticia:
Iran  #39;committed #39; to nuke freeze Iran #39;s chief negotiator to the United Nations nuclear agency says Tehran is committed to the freeze on uranium enrichment agreed earlier this month.
tema correcto: World


### **Baseline**

Tres números sobre las 250 noticias de evaluación, con generación greedy, la distancia
entre validez estricta y tolerante es la parte del problema que son solo las cercas


In [ ]:
#Cargamos el modelo base, sin ajustar nada
base = cargar_base()

print("clasificando las noticias de evaluación...")
SAL_ANTES = generar_lote(base, TEXTO_EVAL)

VAL_ANTES = [valido(s) for s in SAL_ANTES]
TOL_ANTES = [valido_tolerante(s) for s in SAL_ANTES]
ACI_ANTES = [int(tema_de(s) == t) for s, t in zip(SAL_ANTES, TEMA_EVAL)]

print(f"validez estricta (json.loads crudo): {sum(VAL_ANTES) / len(VAL_ANTES):.3f}")
print(f"validez tolerante (sin cercas): {sum(TOL_ANTES) / len(TOL_ANTES):.3f}")
print(f"exactitud del tema: {sum(ACI_ANTES) / len(ACI_ANTES):.3f}")

print("tres salidas del modelo base:")
for s in SAL_ANTES[:3]:
    print(repr(s[:200]))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

clasificando las noticias de evaluación...
  250/250
validez estricta (json.loads crudo): 0.860
validez tolerante (sin cercas): 0.860
exactitud del tema: 0.780
tres salidas del modelo base:
'{"topic": "World", "headline": "Iran committed to nuke freeze"}'
'{"topic": "World", "headline": "Dollar Rises on Interest Rate Advantage"}'
'{"topic":"Business","headline":"Air Canada appoints Montie Brewer as CEO"}'


### **Los pares, enteros del propio modelo**

Tres intentos por noticia a temperatura 0.8, de cada noticia con al menos:
* un intento válido
* uno inválido

sale un par

Es importante para el experimento que los dos lados salgan de la misma política, si uno viniera de otra fuente, el par diferiría en muchas cosas y el modelo aprendería la más fácil.

In [ ]:
print(f"{K_INTENTOS} intentos por noticia, temperatura 0.8...")
INTENTOS = []
for k in range(K_INTENTOS):
    print(f"  intento {k + 1} de {K_INTENTOS}")
    INTENTOS.append(generar_lote(base, TEXTO_TRAIN, temperatura=0.8, semilla=1000 * (k + 1)))

3 intentos por noticia, temperatura 0.8...
  intento 1 de 3
  300/300
  intento 2 de 3
  300/300
  intento 3 de 3
  300/300


Solo sirven las noticias mezcladas: al menos un intento válido y al menos uno inválido
* si los tres salieron bien no hay nada que enseñar
* si los tres salieron mal no hay lado elegido
* El len(t.strip()) > 5 descarta salidas vacías o casi vacías, que enseñarían "no digas nada" en lugar de "escribe JSON bien formado"

In [ ]:
rnd = random.Random(11)
pares, siempre_val, nunca_val = [], 0, 0
for i, texto in enumerate(TEXTO_TRAIN):
    intentos = [INTENTOS[k][i] for k in range(K_INTENTOS)]
    buenos = [t for t in intentos if valido(t)]
    malos = [t for t in intentos if not valido(t) and len(t.strip()) > 5]
    if buenos and malos:
        pares.append({"prompt": prompt_de(texto),
                      "chosen": rnd.choice(buenos),
                      "rejected": rnd.choice(malos)})
    elif buenos:
        siempre_val += 1
    else:
        nunca_val += 1

DATOS = Dataset.from_list(pares)
print(f"de {len(TEXTO_TRAIN)} noticias:")
print(f"siempre válidas (no enseñan)  : {siempre_val}")
print(f"nunca válidas (no hay elegida): {nunca_val}")
print(f"mezcladas: pares utilizables: {len(DATOS)}")

de 300 noticias:
siempre válidas (no enseñan)  : 184
nunca válidas (no hay elegida): 24
mezcladas: pares utilizables: 92


Un par, dos intentos del mismo modelo sobre la misma noticia

In [ ]:
if pares:
    print("Elegida (json.loads la acepta)")
    print(repr(pares[0]["chosen"][:260]))

    print("Descartada (se cae)")
    print(repr(pares[0]["rejected"][:260]))

Elegida (json.loads la acepta)
'{"topic": "Sci/Tech", "headline": "New Secure Web Browser Released"}'
Descartada (se cae)
'{"topic": Sci/Tech", "headline": "New Secure Browser Launched as IE and Firefox Compete"}'


### **Entrenar**

In [ ]:
import dataclasses, inspect
from trl import DPOConfig, DPOTrainer

Esto es para sobrevivir a los cambios de API de TRL:

* Si el objetivo es una dataclass (DPOConfig), lee sus campos
* Si es una clase normal (DPOTrainer), lee la firma de __init__
* Si acepta **kwargs, pasa todo sin filtrar, porque ahí no se puede saber
* Devuelve los argumentos aceptados y la lista de los descartados, que se imprime

En esta corrida salió DPOConfig ignora: max_prompt_length y DPOTrainer ignora: tokenizer

In [ ]:
def solo_los_que_acepta(objetivo, deseados):
    if dataclasses.is_dataclass(objetivo):
        validos = {f.name for f in dataclasses.fields(objetivo)}
    else:
        params = inspect.signature(objetivo.__init__).parameters
        if any(p.kind == p.VAR_KEYWORD for p in params.values()):
            return dict(deseados), []
        validos = set(params) - {"self"}
    aceptados = {k: v for k, v in deseados.items() if k in validos}
    return aceptados, sorted(set(deseados) - set(aceptados))

In [ ]:
LORA = LoraConfig(r=16,
                  lora_alpha=32,
                  lora_dropout=0.05,
                  target_modules=["q_proj",
                                  "k_proj",
                                  "v_proj",
                                  "o_proj",
                                  "gate_proj",
                                  "up_proj",
                                  "down_proj"],

                  task_type="CAUSAL_LM", bias="none")

* beta=0.10 controla cuánto se le permite al modelo alejarse de la referencia
* valores bajos dan más libertad y más riesgo de degradar todo lo demás
* valores altos anclan más fuerte
* 0.1 es el valor habitual de la literatura

Loss_type="sigmoid" es DPO clásico

In [ ]:
DESEADOS = dict(
    output_dir="dpo-json",
    beta=0.10,
    loss_type="sigmoid",
    learning_rate=1e-5,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=512,
    max_prompt_length=384,
    fp16=USA_FP16,
    bf16=USA_BF16,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    seed=11,
)

In [ ]:
args_ok, ign = solo_los_que_acepta(DPOConfig, DESEADOS)
if ign:
  print("DPOConfig ignora:", ", ".join(ign))

lote = DESEADOS["per_device_train_batch_size"] * DESEADOS["gradient_accumulation_steps"]

print(f"{len(DATOS)} pares · lote efectivo {lote} · "
      f"~{max(1, len(DATOS) // lote) * DESEADOS['num_train_epochs']} pasos")

DPOConfig ignora: max_prompt_length
92 pares · lote efectivo 8 · ~22 pasos


* **prepare_model_for_kbit_training** deja en fp32 las capas sensibles (normalizaciones, embeddings), habilita gradient checkpointing y prepara las entradas para que el gradiente fluya a través de los pesos cuantizados
* **ref_model=None** es el truco, DPO necesita dos modelos: la política y la referencia congelada, con LoRA no hace falta cargar dos: la referencia es este mismo modelo con el adaptador desactivado

In [ ]:
modelo = prepare_model_for_kbit_training(cargar_base())

argumentos = dict(model=modelo,
                  ref_model=None,
                  args=DPOConfig(**args_ok),
                  train_dataset=DATOS,
                  peft_config=LORA,
                  processing_class=tok,
                  tokenizer=tok)

ok, ign = solo_los_que_acepta(DPOTrainer, argumentos)
if "processing_class" in ok:
  ok.pop("tokenizer", None)

if ign:
  print("DPOTrainer ignora:", ", ".join(ign))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

DPOTrainer ignora: tokenizer


In [ ]:
entrenador = DPOTrainer(**ok)
entrenador.train()

Adding EOS to train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,0.506870
20,0.341932


TrainOutput(global_step=24, training_loss=0.3936291957894961, metrics={'train_runtime': 417.7122, 'train_samples_per_second': 0.44, 'train_steps_per_second': 0.057, 'total_flos': 463763706667008.0, 'train_loss': 0.3936291957894961, 'entropy': 0.351812763937882, 'num_tokens': 54810.0, 'logits/chosen': -2.222874409303448, 'logits/rejected': -1.9826475259778449, 'mean_token_accuracy': 0.8639576775687081, 'rewards/chosen': 0.07986654727054494, 'rewards/rejected': -1.4731262241091048, 'rewards/accuracies': 0.9642857142857143, 'rewards/margins': 1.5529927441052027, 'logps/chosen': -7.529904910496303, 'logps/rejected': -24.04392637525286, 'epoch': 2.0})

In [ ]:
h = [r for r in entrenador.state.log_history if "rewards/margins" in r]
fin = h[-1] if h else {}

print("cómo terminó el entrenamiento:")
for k in ["rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"]:
    if k in fin: print(f"  {k:22s} {fin[k]:+.4f}")

cómo terminó el entrenamiento:
  rewards/chosen         +0.0799
  rewards/rejected       -1.4731
  rewards/margins        +1.5530
  rewards/accuracies     +0.9643


Estas "recompensas" no son de un modelo de recompensa entrenado. Son la recompensa implícita de DPO, es decir, cuanto subio o bajó la log-probabilidad de esa salida respecto de la referencia, escalada por beta

Estos numeros no son el resultado, el resultado es la validez, una celda mas abajo

In [ ]:
entrenador.model.save_pretrained("adaptador-json")

### **Medimos de nuevo**

Mismas 250 noticias, mismo orden y misma generación greedy, lo único distinto es el adaptador


Gradient checkpointing es incompatible con la caché de atención, durante el entrenamiento se desactiva use_cache y hay que volver a prenderla, si no la generación es varias veces más lenta

In [ ]:
entrenado = entrenador.model
entrenado.eval()
if hasattr(entrenado, "gradient_checkpointing_disable"):
    entrenado.gradient_checkpointing_disable()
entrenado.config.use_cache = True

clasificando las mismas noticias con el adaptador...

In [ ]:
SAL_DESPUES = generar_lote(entrenado, TEXTO_EVAL)
VAL_DESPUES = [valido(s) for s in SAL_DESPUES]
TOL_DESPUES = [valido_tolerante(s) for s in SAL_DESPUES]
ACI_DESPUES = [int(tema_de(s) == t) for s, t in zip(SAL_DESPUES, TEMA_EVAL)]

  250/250


In [ ]:
print("tres salidas del modelo entrenado:")
for s in SAL_DESPUES[:3]:
    print("\n" + repr(s[:200]))

tres salidas del modelo entrenado:

'{"topic":"World","headline":"Iran committed to nuke freeze"}'

'{"topic":"World","headline":"Dollar Rises on Interest Rate Plays"}'

'{"topic":"Business","headline":"Air Canada appoints Montie Brewer as CEO"}'


### **tres casos que el adaptador arregló**


In [ ]:
arreglados = [i for i in range(len(TEXTO_EVAL))
              if VAL_ANTES[i] == 0 and VAL_DESPUES[i] == 1][:3]

if not arreglados:
    print("no hubo casos que pasaran de inválido a válido; se muestran tres cualquiera")
    arreglados = [0, 1, 2]

for i in arreglados:
    print("\n" + "=" * 78)
    print("Noticia:", TEXTO_EVAL[i][:220])
    print("tema correcto:", TEMA_EVAL[i])
    print("=" * 78)
    print(f"  ANTES   [{'válido' if VAL_ANTES[i] else 'INVÁLIDO'}] {SAL_ANTES[i][:180]!r}")
    print(f"  DESPUÉS [{'válido' if VAL_DESPUES[i] else 'INVÁLIDO'}] {SAL_DESPUES[i][:180]!r}")


NOTICIA: Toshiba to use perpendicular recording in new HDDs Toshiba is close to commercializing a new data storage technology that could significantly increase the capacity of hard-disk drives, it said Tuesday.
tema correcto: Sci/Tech
  ANTES   [INVÁLIDO] '{"topic": Sci/Tech, "headline": Toshiba to use perpendicular recording in new HDDs}'
  DESPUÉS [válido] '{"topic":"Sci/Tech","headline":"Toshiba to Use Perpendicular Recording in New HDDs"}'

NOTICIA: Spain hunts for fans who racially abused England players The Spanish government responded to diplomatic pressure from Britain yesterday by starting a search for fans who racially abused England players during a  quot;fri
tema correcto: Sports
  ANTES   [INVÁLIDO] '{"topic": World, "headline": "Spanish Government Starts Search for Fans Who Racially Abused English Players"}'
  DESPUÉS [válido] '{"topic":"World","headline":"Spanish Government Starts Search for Fans Who Racially Abused English Players"}'

NOTICIA: EU will go it alone on nu

Entonces...

- **DPO entrena con pares ordenados, no con respuestas ideales**, nadie  escribió el JSON correcto, se ordenaron de a dos intentos del propio modelo

- **Los dos lados del par salen de la misma política**, así que lo único que el modelo puede aprender es lo que los distingue: si un programa puede leer la salida

- **La mejora se corrige**, `json.loads` no tiene opinión


**Que tan importante es esto en un pipeline?**

* Un modelo dentro de una pipeline no interactua con una persona.
* Las cercas de markdown y las frases de cortesía no son un defecto de estilo: son errores
* Cómo el criterio de éxito es un `json.loads`, la señal de preferencia se puede recolectar sola, sin anotadores

**Lo que falta**

En un sistema real la preferencia se mueve y no hay `json.loads` que la mida:
* cómo se detecta que la preferencia cambió si nadie lo avisa?
* con qué evidencia se decide que la versión nueva puede reemplazar a la que está sirviendo?
* qué pasa cuando el modelo mejora en la métrica y empeora en todo lo demás?
